# Setup

In [82]:
!pip install -qqU deepeval datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.7/558.7 kB 19.3 MB/s eta 0:00:00


- deepeval - narzędzie do oceny modeli głębokiego uczenia, pozwalające mierzyć ich skuteczność i jakość działania
- datasets - kompleksowa biblioteka od Hugging Face, która udostępnia setki gotowych zbiorów danych do trenowania modeli uczenia maszynowego oraz narzędzia do ich efektywnego przetwarzania

In [ ]:
import os
from google.colab import userdata
from deepeval.metrics import (
    ConversationalGEval,
    RoleAdherenceMetric,
    KnowledgeRetentionMetric,
    ConversationCompletenessMetric,
    ConversationRelevancyMetric,
)
from deepeval.test_case import LLMTestCase, ConversationalTestCase
from datasets import load_dataset

In [ ]:
class CFG:
    model = "gpt-4o-mini"
    temp = 0.3
    dataset = "flpelerin/ChatAlpaca-10k"

In [ ]:
# setup OpenAI connection
api_key = userdata.get("openaivision")
os.environ["OPENAI_API_KEY"] = api_key

# Funkcje

In [ ]:
# largest even number no greater than k
def laevnu(k):
    if k % 2 == 0:
        return k
    return k - 1

Ta funkcja `laevnu` (skrót od "largest even number no greater than k") znajduje największą liczbę parzystą nie większą od podanej liczby k. Przeanalizujmy jej działanie:

Funkcja przyjmuje jeden parametr k i sprawdza jego parzystość za pomocą operacji modulo (%). Kiedy dzielimy liczbę przez 2, reszta z dzielenia (%) może wynosić 0 lub 1. Jeśli reszta wynosi 0, liczba jest parzysta, a jeśli 1 - nieparzysta.

W przypadku gdy k jest liczbą parzystą (k % 2 == 0), funkcja zwraca bezpośrednio wartość k, ponieważ jest to już największa liczba parzysta nie większa od k.

Natomiast jeśli k jest liczbą nieparzystą, funkcja odejmuje od niej 1 (k - 1), otrzymując w ten sposób największą liczbę parzystą mniejszą od k.

Na przykład:
- Dla k = 6 (liczba parzysta) → funkcja zwróci 6
- Dla k = 7 (liczba nieparzysta) → funkcja zwróci 6
- Dla k = 0 (liczba parzysta) → funkcja zwróci 0
- Dla k = -3 (liczba nieparzysta) → funkcja zwróci -4

In [ ]:
def create_list_of_test_cases(dd):
    test_cases = []
    for jj in range(0, laevnu(len(dd)), 2):
        tc = LLMTestCase(input=dd[jj]["value"], actual_output=dd[jj + 1]["value"])
        test_cases.append(tc)

    return test_cases

Ta funkcja tworzy listę przypadków testowych dla modelu językowego na podstawie dostarczonych danych. Przeanalizujmy jej działanie krok po kroku:

Funkcja przyjmuje parametr dd, który najprawdopodobniej jest listą słowników zawierających pary pytań i odpowiedzi w formacie JSON. Każdy element tej listy ma klucz 'value' przechowujący treść wiadomości.

Na początku tworzymy pustą listę test_cases, która będzie gromadzić nasze przypadki testowe. Jest to standardowa praktyka w Pythonie - zaczynamy od pustego kontenera i stopniowo go wypełniamy.

Pętla for iteruje przez indeksy listy dd z krokiem 2, zaczynając od 0. Używamy tu wcześniej zdefiniowanej funkcji laevnu, aby upewnić się, że nie wyjdziemy poza zakres listy - bierzemy największą parzystą liczbę nie większą od długości listy dd. To zabezpieczenie jest istotne, ponieważ będziemy zawsze potrzebować par elementów (pytanie i odpowiedź).

W każdej iteracji tworzymy nowy przypadek testowy (LLMTestCase) z dwóch kolejnych elementów listy:
- dd[jj]['value'] staje się danymi wejściowymi (input) - to prawdopodobnie pytanie lub prompt
- dd[jj+1]['value'] staje się oczekiwanym wynikiem (actual_output) - to prawdopodobnie wzorcowa odpowiedź

Krok co 2 w pętli (range(..., 2)) jest kluczowy - zapewnia, że bierzemy elementy parami. Na przykład, jeśli dd ma elementy [A,B,C,D], to utworzymy przypadki testowe z par (A,B) i (C,D).

Każdy utworzony przypadek testowy jest dodawany do listy test_cases za pomocą metody append. Na końcu funkcja zwraca kompletną listę przypadków testowych.

# Dane

In [ ]:
# load the conversational dataset
dataset = load_dataset(CFG.dataset, split="train")

In [ ]:
dataset[0]["conversations"]

[{'from': 'human', 'value': 'Find the product of the numbers: 5 and 8'},
 {'from': 'gpt', 'value': 'The product of 5 and 8 is 40.'},
 {'from': 'human', 'value': 'What is the sum of the numbers 6 and 12?'},
 {'from': 'gpt', 'value': 'The sum of the numbers 6 and 12 is 18.'},
 {'from': 'human', 'value': 'Can you tell me the quotient of 20 and 5?'},
 {'from': 'gpt', 'value': 'Yes, the quotient of 20 and 5 is 4.'},
 {'from': 'human', 'value': 'What is the difference between 25 and 13?'},
 {'from': 'gpt', 'value': 'The difference between 25 and 13 is 12.'},
 {'from': 'human', 'value': 'What is the square of 9?'},
 {'from': 'gpt', 'value': 'The square of 9 is 81.'},
 {'from': 'human', 'value': 'What is the cube of 6?'},
 {'from': 'gpt', 'value': 'The cube of 6 is 216.'}]

In [ ]:
dataset[12]["conversations"]

[{'from': 'human',
  'value': 'Find the angle in the triangle using trigonometry: AB=5, BC=7, AC=9'},
 {'from': 'gpt', 'value': 'Angle A = 65.7 degrees'},
 {'from': 'human', 'value': 'What is the value of Angle B in the triangle?'},
 {'from': 'gpt',
  'value': 'To find the value of angle B in the triangle using trigonometry, we can use the Law of Cosines:\n\ncos(B) = (a^2 + c^2 - b^2) / 2ac\n\nwhere a, b, and c are the lengths of the sides opposite to angles A, B, and C, respectively.\n\nIn this case, we have:\n\na = 5 (opposite to angle A)\nb = ?\nc = 7 (opposite to angle B)\n\nWe can use the Law of Cosines again to find the value of b:\n\nb^2 = a^2 + c^2 - 2ac cos(B)\nb^2 = 5^2 + 7^2 - 2(5)(7) cos(B)\nb^2 = 25 + 49 - 70 cos(B)\nb^2 = 74 - 70 cos(B)\n\nNow we can substitute this expression for b^2 into the first equation:\n\ncos(B) = (a^2 + c^2 - b^2) / 2ac\ncos(B) = (5^2 + 7^2 - (74 - 70 cos(B))) / (2)(5)(7)\ncos(B) = (74 - 70 cos(B)) / 70\n70 cos(B) = 74 - cos(B)\n71 cos(B) = 74\nco

In [ ]:
dd = dataset[0]["conversations"]
dd

[{'from': 'human', 'value': 'Find the product of the numbers: 5 and 8'},
 {'from': 'gpt', 'value': 'The product of 5 and 8 is 40.'},
 {'from': 'human', 'value': 'What is the sum of the numbers 6 and 12?'},
 {'from': 'gpt', 'value': 'The sum of the numbers 6 and 12 is 18.'},
 {'from': 'human', 'value': 'Can you tell me the quotient of 20 and 5?'},
 {'from': 'gpt', 'value': 'Yes, the quotient of 20 and 5 is 4.'},
 {'from': 'human', 'value': 'What is the difference between 25 and 13?'},
 {'from': 'gpt', 'value': 'The difference between 25 and 13 is 12.'},
 {'from': 'human', 'value': 'What is the square of 9?'},
 {'from': 'gpt', 'value': 'The square of 9 is 81.'},
 {'from': 'human', 'value': 'What is the cube of 6?'},
 {'from': 'gpt', 'value': 'The cube of 6 is 216.'}]

In [ ]:
dataset[12]["conversations"]

[{'from': 'human',
  'value': 'Find the angle in the triangle using trigonometry: AB=5, BC=7, AC=9'},
 {'from': 'gpt', 'value': 'Angle A = 65.7 degrees'},
 {'from': 'human', 'value': 'What is the value of Angle B in the triangle?'},
 {'from': 'gpt',
  'value': 'To find the value of angle B in the triangle using trigonometry, we can use the Law of Cosines:\n\ncos(B) = (a^2 + c^2 - b^2) / 2ac\n\nwhere a, b, and c are the lengths of the sides opposite to angles A, B, and C, respectively.\n\nIn this case, we have:\n\na = 5 (opposite to angle A)\nb = ?\nc = 7 (opposite to angle B)\n\nWe can use the Law of Cosines again to find the value of b:\n\nb^2 = a^2 + c^2 - 2ac cos(B)\nb^2 = 5^2 + 7^2 - 2(5)(7) cos(B)\nb^2 = 25 + 49 - 70 cos(B)\nb^2 = 74 - 70 cos(B)\n\nNow we can substitute this expression for b^2 into the first equation:\n\ncos(B) = (a^2 + c^2 - b^2) / 2ac\ncos(B) = (5^2 + 7^2 - (74 - 70 cos(B))) / (2)(5)(7)\ncos(B) = (74 - 70 cos(B)) / 70\n70 cos(B) = 74 - cos(B)\n71 cos(B) = 74\nco

# Metryki


In [ ]:
dlist = create_list_of_test_cases(dataset[12]["conversations"])

## Role adherence

In [ ]:
#
convo_test_case = ConversationalTestCase(
    chatbot_role="You are a helpful and polite assistant", turns=dlist
)

metric = RoleAdherenceMetric(threshold=0.5)

metric.measure(convo_test_case)
print(metric.score)
metric.reason

Output()

Event loop is already running. Applying nest_asyncio patch to allow async execution...

False @

0.3333333333333333


'The score is 0.3333333333333333 because the LLM chatbot responses in turns #2 and #3 deviated significantly from the role of a "helpful and polite assistant." \n\nIn turn #2, the response given is: \'To find the value of angle B in the triangle using trigonometry, we can use the Law of Cosines... B = 21.37 degrees.\' This response is overly technical and lengthy, making it difficult for a layperson to understand quickly. It lacks the conduciveness of a polite assistant by not simplifying or breaking down the explanation in a more approachable manner, potentially overwhelming the user.\n\nIn turn #3, the response \'To find the value of angle C in the triangle using trigonometry, we can use the Law of Cosines... C = 29.1 degrees.\' not only continues to offer intricate explanations but also contains a calculation error. This deviation results in the bot being perceived as unhelpful and inaccurate, contrasting the expected behavior of a helpful role. These two instances severely affected

## Knowledge retention

In [ ]:
test_case = ConversationalTestCase(turns=dlist)
metric = KnowledgeRetentionMetric(threshold=0.5)

metric.measure(test_case)
print(metric.score)
metric.reason

Output()

Event loop is already running. Applying nest_asyncio patch to allow async execution...

1.0


'The score is 1.00 because there are no attritions, indicating perfect retention of knowledge throughout the conversation.'

## Conversation completeness

In [ ]:
test_case = ConversationalTestCase(turns=dlist)
metric = ConversationCompletenessMetric(threshold=0.5)

metric.measure(convo_test_case)
print(metric.score)
metric.reason

Output()

Event loop is already running. Applying nest_asyncio patch to allow async execution...

0.5


"The score is 0.5 because the LLM response partially meets the user's intention by calculating angles A and C using trigonometry. However, it falls short of the user's request for a detailed and accurate step-by-step approach, specifically for angle B. The response included inconsistencies in calculations for angle B, such as 'mixing up known lengths,' which undermines the reliability of solving all angles as the user desired."

## Conversation relevancy

In [ ]:
test_case = ConversationalTestCase(turns=dlist)
metric = ConversationRelevancyMetric(threshold=0.5)

metric.measure(convo_test_case)
print(metric.score)
metric.reason

Output()

Event loop is already running. Applying nest_asyncio patch to allow async execution...

0.6666666666666666


"The score is 0.67 because message number 3 contains an incorrect approach to calculating angle C in a triangle, using inconsistent values for sides 'a' and 'b'. This irrelevance stems from using the Pythagorean theorem incorrectly, leading to an erroneous solution for angle C, impacting the overall relevance of the actual outputs."